Objective: Load preprocessed 3D EEG epoch tensor $X \in \mathbb{R}^{N \times 64 \times 641}$ and binary target vector $y \in \{0, 1\}^N$.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import mne
from scipy import linalg
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold


In [5]:
sys.path

['/opt/pyenv/versions/3.13.1/lib/python313.zip',
 '/opt/pyenv/versions/3.13.1/lib/python3.13',
 '/opt/pyenv/versions/3.13.1/lib/python3.13/lib-dynload',
 '',
 '/home/jvalenci/Desktop/Total-Perspective-Vortex/.env/lib/python3.13/site-packages',
 '/home/jvalenci/Desktop/Total-Perspective-Vortex/srcs']

In [8]:
# abspath returns of a given relative path
lMiscPath = os.path.abspath('../srcs')

In [11]:

# Add source directory to path to access local modules
if lMiscPath not in sys.path:
    sys.path.append(os.path.abspath('../srcs'))

from misc import load_and_parse_eeg


In [ ]:
# Suppress verbose MNE logs for clarity
mne.set_log_level('WARNING')

# 1. Configuration: 102-Subject Benchmark Cohort
BASE_DATA_PATH = "mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"
ANOMALY_EXCLUDED_SUBJECTS = [38, 88, 89, 92, 100, 104, 106]
CLEAN_SUBJECTS = [s for s in range(1, 110) if s not in ANOMALY_EXCLUDED_SUBJECTS]
RUN_IDS = [4, 8, 12]  # Unilateral Motor Imagery: Left vs. Right Fist

In [12]:
# 2. Data Ingestion
X, y, df_metadata = load_and_parse_eeg(
    subject_ids=CLEAN_SUBJECTS,
    run_ids=RUN_IDS,
    base_path=BASE_DATA_PATH,
    tmin=0.0,
    tmax=4.0,
    include_rest=False
)

# 3. Input Verification & Visual Inspection
print(f"X Type: {type(X)} | Shape: {X.shape} | Dtype: {X.dtype}")
print(f"y Type: {type(y)} | Shape: {y.shape} | Dtype: {y.dtype}")
print(f"Target Class Counts: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Signal Statistics: Min={X.min():.2e}, Max={X.max():.2e}, Mean={X.mean():.2e}, Std={X.std():.2e}")

Processing Subjects:   0%|          | 0/102 [00:00<?, ?it/s]

Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 mon

#### **Objective:**  This mesures how nodes interact with each other during each type of thinking task (e.g., left hand vs right hand motor imagery) by using the covariance matrices of the EEG signals.

* The covariance matrices are computed for each trial and then averaged across trials for each class. The resulting mean covariance matrices are then normalized to ensure that they have unit trace, which is a common practice in EEG analysis to facilitate comparison between different conditions.

* Compute normalized mean spatial covariance matrices $\Sigma_0, \Sigma_1 \in \mathbb{R}^{64 \times 64}$ for Class 0 ($y=0$) and Class 1 ($y=1$) from input trials $X_i \in \mathbb{R}^{64 \times 641}$.

for deep understanding see the following references:

https://app.notion.com/p/jvalenci/total-perspective-vortex-3929d52658e080088fdcc42b60719b01?source=copy_link#3e69d52658e080ec9aefd5f088a1cd80

In [ ]:
# 1. Trial-wise Normalized Covariance Estimation: Sigma = (X * X.T) / trace(X * X.T)
def compute_class_covariances(X, y):
    """Computes mean normalized spatial covariance for each binary class."""
    covs = []
    for target in [0, 1]:
        # !Filter trials for specific class
        # e.g., X_class shape: (N_trials_class, C, T)
        # data structure y is a 1D array of shape (N_trials,) containing class labels
        X_class = X[y == target]

        # Batch compute covariance for each trial: (N_trials, C, T) -> (N_trials, C, C)
        # Using np.matmul for trial-wise dot product: X_i @ X_i.T
        trial_covs = np.matmul(X_class, X_class.transpose(0, 2, 1))

        # Normalize each trial covariance by its trace to ensure unit variance across sensors
        # np.einsum provides a efficient way to extract the diagonal and sum for batch trace
        traces = np.einsum("ijj->i", trial_covs)
        normalized_trial_covs = trial_covs / traces[:, np.newaxis, np.newaxis]

        # Average across trials to get class-conditional mean covariance
        covs.append(np.mean(normalized_trial_covs, axis=0))
    return covs[0], covs[1]


Sigma_0, Sigma_1 = compute_class_covariances(X, y)

# 2. Statistical Verification & Symmetry Check
print(f"Sigma_0 Shape: {Sigma_0.shape} | Sigma_1 Shape: {Sigma_1.shape}")
print(f"Sigma_0 Dtype: {Sigma_0.dtype} | Sigma_1 Dtype: {Sigma_1.dtype}")
print(
    f"Symmetry Check (Max Diff Σ - Σ.T): Class 0 = {np.max(np.abs(Sigma_0 - Sigma_0.T)):.2e}, Class 1 = {np.max(np.abs(Sigma_1 - Sigma_1.T)):.2e}"
)
print(
    f"Trace Verification (Should be 1.0): Class 0 = {np.trace(Sigma_0):.2f}, Class 1 = {np.trace(Sigma_1):.2f}"
)

Sigma_0 Shape: (64, 64) | Sigma_1 Shape: (64, 64)
Sigma_0 Dtype: float64 | Sigma_1 Dtype: float64
Symmetry Check (Max Diff Σ - Σ.T): Class 0 = 0.00e+00, Class 1 = 0.00e+00
Trace Verification (Should be 1.0): Class 0 = 1.00, Class 1 = 1.00
